# Notebook 15 — Packaging with pyproject.toml

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 15 of 20  
**Prerequisites:** Notebook 14  
**Time estimate:** 60 minutes

---
## Step 1 — Motivation

A Python package that cannot be installed with a single `pip install -e .` is not
truly reusable. The `pyproject.toml` file is the modern Python standard for declaring
a package's metadata, dependencies, and build configuration.

By the end of this notebook, `compbio_utils` version `0.1.0` will be correctly
versioned, installable, and citeable.

---
## Step 2 — Intuition

`pyproject.toml` replaced `setup.py` and `setup.cfg`. It is the single authoritative
file that answers: what is this package, who made it, what does it need to run,
and how do I build and install it?

The three sections you always need: `[build-system]`, `[project]`, and (optionally)
`[tool.ruff]`, `[tool.pytest.ini_options]`.

---
## Step 3 — Biological Background

**Why packaging matters for Track A and B:**

- **Track A**: a bioinformatics engineering role expects you to know how to distribute
  a tool to a cluster. `pip install compbio_utils` is how that happens.
- **Track B**: a cited, versioned, DOI-bearing package is a portfolio artifact. ZENODO
  and PyPI both require proper packaging.

**CITATION.cff**: a standardized machine-readable citation file. GitHub automatically
uses it to generate a 'Cite this repository' button. Adding one is a 5-minute task
with measurable impact on academic discoverability.

---
## Step 4 — Mathematical Explanation

Not applicable — this notebook is about packaging infrastructure.

---
## Step 5 — Computational Explanation

**`pyproject.toml` key sections:**

```toml
[build-system]    ← tells pip which build tool to use (hatchling, flit, setuptools)
[project]         ← name, version, description, requires-python, dependencies
[project.optional-dependencies]  ← dev/test extras (pip install -e .[dev])
[tool.ruff]       ← linting configuration
[tool.pytest.ini_options]  ← test discovery, markers
```

**Semantic versioning reminder:** `0.1.0`  
- Major (`0.x.x`): breaking API changes  
- Minor (`x.1.x`): new features, backward-compatible  
- Patch (`x.x.0`): bug fixes

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — Write the final compbio_utils pyproject.toml
import pathlib

repo_root = pathlib.Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists(): break
    repo_root = repo_root.parent

pyproject_content = '''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "compbio-utils"
version = "0.1.0"
description = "Computational biology utilities — built during the 12-month Research Accelerator"
readme = "README.md"
requires-python = ">=3.11"
license = {text = "MIT"}
authors = [
    {name = "Vinoth", email = "vinoth.ac.in@gmail.com"}
]
keywords = ["computational-biology", "bioinformatics", "genomics", "sequence-analysis"]
classifiers = [
    "Programming Language :: Python :: 3",
    "License :: OSI Approved :: MIT License",
    "Intended Audience :: Science/Research",
]
dependencies = [
    "numpy>=1.26",
    "scipy>=1.12",
    "pandas>=2.2",
    "matplotlib>=3.8",
    "seaborn>=0.13",
    "biopython>=1.83",
    "scikit-learn>=1.4",
]

[project.optional-dependencies]
dev = [
    "pytest>=8.0",
    "pytest-cov",
    "ruff>=0.4",
]

[tool.ruff]
line-length = 88
target-version = "py311"
select = ["E", "F", "W", "I", "N", "B"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"
markers = [
    "slow: marks tests as slow (deselect with -m 'not slow')",
]
'''

pyproject_path = repo_root / "utilities" / "compbio_utils" / "pyproject.toml"
if pyproject_path.exists():
    print(f"pyproject.toml already exists at {pyproject_path}")
    print("Review and merge any differences manually.")
    print("\nTarget content:")
    print(pyproject_content)
else:
    pyproject_path.write_text(pyproject_content.strip())
    print(f"Written: {pyproject_path}")

In [ ]:
# Cell 6.2 — CITATION.cff
citation_content = '''cff-version: 1.2.0
message: "If you use this software, please cite it as below."
type: software
title: "compbio-utils: Computational Biology Utilities"
version: "0.1.0"
authors:
  - name: Vinoth
    email: vinoth.ac.in@gmail.com
repository-code: "https://github.com/Vinoth-ai-20/computational-biology"
license: MIT
'''

citation_path = repo_root / "utilities" / "compbio_utils" / "CITATION.cff"
if not citation_path.exists():
    citation_path.write_text(citation_content)
    print(f"Written: {citation_path}")
else:
    print(f"CITATION.cff already exists: {citation_path}")

In [ ]:
# Cell 6.3 — Verify the package installs cleanly
import subprocess, sys

cbu_path = repo_root / "utilities" / "compbio_utils"
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(cbu_path), "--quiet"],
    capture_output=True, text=True
)
print("Install stdout:", result.stdout or "(none)")
print("Install stderr:", result.stderr[:500] if result.stderr else "(none)")
print(f"Return code: {result.returncode}")

# Verify
import importlib
importlib.invalidate_caches()
try:
    import compbio_utils
    print(f"\ncompbio_utils {compbio_utils.__version__} imported successfully")
except Exception as e:
    print(f"\nImport failed: {e}")

In [ ]:
# Cell 6.4 — Tag v0.1.0 in git
# Do NOT run this automatically — run it manually after confirming tests pass.
print("When ready, run these commands in the terminal:")
print()
print("  cd utilities/compbio_utils")
print("  git add .")
print("  git commit -m 'module-01/packaging: compbio_utils v0.1.0'")
print("  git tag -a compbio-utils-v0.1.0 -m 'Module 01 milestone: compbio_utils v0.1.0'")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Package metadata summary
try:
    import importlib.metadata as meta
    dist = meta.metadata("compbio-utils")
    print("Package metadata:")
    for key in ["Name", "Version", "Summary", "Requires-Python", "License"]:
        print(f"  {key:<18} {dist.get(key, 'N/A')}")
except importlib.metadata.PackageNotFoundError:
    print("compbio-utils not installed — complete Cell 6.3 first.")

---
## Step 8 — Exercises

1. Run `pip show compbio-utils` and verify all fields are correct.
2. Add `hatchling` to `requirements.txt` if it's not there.
3. Run `pip install -e utilities/compbio_utils[dev]` and verify `pytest` and `ruff` resolve.
4. What is the difference between `version = "0.1.0"` in `pyproject.toml` and
   `__version__ = "0.1.0"` in `__init__.py`? Which is the single source of truth?

---
## Quiz — Active Recall

1. What is the difference between `pip install compbio-utils` and `pip install -e compbio-utils`?
2. What does `[project.optional-dependencies]` allow?
3. What are the three fields in semantic versioning and what change triggers each?
4. What is CITATION.cff and why does it matter for academic software?
5. What does `hatchling` do? Name one alternative.

---
## Reflection

**Date completed:** ____________________

> *[Did pip install succeed? Is the version tag in git? Is CITATION.cff written?]*

---
*Next: `16_biopython_fundamentals.ipynb`*